In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import cv2
import random
import pandas as pd
from tqdm import tqdm

In [4]:
base_path = "/kaggle/input/datasets/organizations/airbusgeo/airbus-oil-storage-detection-dataset"

print(os.listdir(base_path))

image_dir = os.path.join(base_path, "images")
csv_path = os.path.join(base_path, "annotations.csv")

df = pd.read_csv(csv_path)
df.head()

['README.md', 'LICENSE.txt', 'images', 'annotations.csv', 'extras']


,image_id,class,bounds
0,0918a15c-8cfb-4da7-92cf-1f5098a76760,oil-storage-tank,"(1205, 1493, 1231, 1516)"
1,0918a15c-8cfb-4da7-92cf-1f5098a76760,oil-storage-tank,"(1479, 1756, 1507, 1776)"
2,0918a15c-8cfb-4da7-92cf-1f5098a76760,oil-storage-tank,"(1478, 1786, 1518, 1815)"
3,0918a15c-8cfb-4da7-92cf-1f5098a76760,oil-storage-tank,"(2132, 2163, 2148, 2178)"
4,0918a15c-8cfb-4da7-92cf-1f5098a76760,oil-storage-tank,"(2010, 2127, 2024, 2141)"


In [6]:
def parse_bounds(bounds_str):
    bounds_str = bounds_str.strip("()")
    x_min, y_min, x_max, y_max = map(int, bounds_str.split(","))
    return [x_min, y_min, x_max, y_max]

In [7]:
def convert_to_yolo(size, bbox):
    w_img, h_img = size

    x_center = (bbox[0] + bbox[2]) / 2 / w_img
    y_center = (bbox[1] + bbox[3]) / 2 / h_img
    w = (bbox[2] - bbox[0]) / w_img
    h = (bbox[3] - bbox[1]) / h_img

    return [x_center, y_center, w, h]

In [8]:
image_ids = df['image_id'].unique().tolist()

random.shuffle(image_ids)
split = int(0.8 * len(image_ids))

train_ids = set(image_ids[:split])
val_ids = set(image_ids[split:])

In [9]:
os.makedirs("tiled/images/train", exist_ok=True)
os.makedirs("tiled/images/val", exist_ok=True)
os.makedirs("tiled/labels/train", exist_ok=True)
os.makedirs("tiled/labels/val", exist_ok=True)

In [10]:
TILE_SIZE = 640
STRIDE = 512
MIN_BOX_SIZE = 10

In [11]:
def tile_image_and_boxes_clean(img, boxes):
    h, w, _ = img.shape
    tiles = []

    for y in range(0, h - TILE_SIZE + 1, STRIDE):
        for x in range(0, w - TILE_SIZE + 1, STRIDE):

            tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
            tile_boxes = []

            for box in boxes:
                x_min, y_min, x_max, y_max = box

                if x_max <= x or x_min >= x + TILE_SIZE:
                    continue
                if y_max <= y or y_min >= y + TILE_SIZE:
                    continue

                new_xmin = max(x_min - x, 0)
                new_ymin = max(y_min - y, 0)
                new_xmax = min(x_max - x, TILE_SIZE)
                new_ymax = min(y_max - y, TILE_SIZE)

                if (new_xmax - new_xmin) < MIN_BOX_SIZE:
                    continue
                if (new_ymax - new_ymin) < MIN_BOX_SIZE:
                    continue

                tile_boxes.append([new_xmin, new_ymin, new_xmax, new_ymax])

            if len(tile_boxes) > 0:
                tiles.append((tile, tile_boxes))

    return tiles

In [12]:
tile_count = 0

for img_id in tqdm(image_ids):
    img_path = os.path.join(image_dir, f"{img_id}.jpg")

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)

    rows = df[df['image_id'] == img_id]
    boxes = [parse_bounds(r['bounds']) for _, r in rows.iterrows()]

    subset = "train" if img_id in train_ids else "val"

    tiles = tile_image_and_boxes_clean(img, boxes)

    for i, (tile, tile_boxes) in enumerate(tiles):

        h, w, _ = tile.shape

        img_name = f"{img_id}_{i}.jpg"
        label_name = f"{img_id}_{i}.txt"

        img_out = f"tiled/images/{subset}/{img_name}"
        label_out = f"tiled/labels/{subset}/{label_name}"

        cv2.imwrite(img_out, tile)

        with open(label_out, "w") as f:
            for box in tile_boxes:
                yolo_box = convert_to_yolo((w, h), box)
                f.write(f"0 {' '.join(map(str, yolo_box))}\n")

        tile_count += 1

print("Total tiles created:", tile_count)

100%|██████████| 98/98 [00:07<00:00, 12.71it/s]

Total tiles created: 821


In [13]:
import glob

labels = glob.glob("tiled/labels/train/*.txt")

non_empty = sum(1 for f in labels if open(f).read().strip())

print("Total tiles:", len(labels))
print("Non-empty labels:", non_empty)

Total tiles: 669
Non-empty labels: 669


In [14]:
yaml_content = """
path: tiled
train: images/train
val: images/val

names:
  0: oil_tank
"""

with open("tiled_dataset.yaml", "w") as f:
    f.write(yaml_content)

In [15]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="tiled_dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    degrees=90,
    flipud=0.5,
    fliplr=0.5
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=tiled_dataset.yaml, degrees=90, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c3b706c33b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [17]:
results = model.predict(
    source="tiled/images/val",
    conf=0.6,        # increase confidence (remove weak boxes)
    iou=0.3,         # stricter NMS (reduce overlaps)
    max_det=300,     # limit max detections per image
    save=True,
    save_txt=True
)


image 1/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_0.jpg: 640x640 8 oil_tanks, 16.3ms
image 2/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_1.jpg: 640x640 1 oil_tank, 16.1ms
image 3/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_2.jpg: 640x640 18 oil_tanks, 16.1ms
image 4/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_3.jpg: 640x640 6 oil_tanks, 16.1ms
image 5/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_4.jpg: 640x640 45 oil_tanks, 16.1ms
image 6/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_5.jpg: 640x640 11 oil_tanks, 16.1ms
image 7/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_6.jpg: 640x640 24 oil_tanks, 16.1ms
image 8/152 /kaggle/working/tiled/images/val/0a75eb88-46ba-4a64-acb3-8919ea880137_7.jpg: 640x640 16 oil_tanks, 16.1ms
image 9/152 /kaggle/working/tiled/images/val/22555a1d-f5c6-